# AMEX Probability of Default (PD) — Local Thesis Pipeline

**Comparative Evaluation of Machine Learning and Deep Learning Models for Predicting Probability of Default**

Cham Ying Chyi (23076054) · University of Malaya · Supervisor: Prof. Dr Rafidah Md Noor

---

This notebook runs the **entire pipeline locally** (built for a Mac Mini M4) in one place:

| Stage | What it does |
|------|--------------|
| 0 | Environment setup & sanity checks |
| 1 | Data landing (CSV → Parquet) & EDA |
| 2 | Preprocessing (denoise, encode `D_63`/`D_64`, drop bad features) |
| 3 | Feature engineering — **snapshot (6 aggregations, ML)** + **sequential top-15 features (DL)** |
| 4 | Feature selection (variance → correlation → LightGBM gain) |
| 5a | ML models: Logistic Regression, XGBoost, LightGBM |
| 5b | DL models: LSTM, GRU, Transformer (**13 epochs**, with timestamps) |
| 5c | Ensembles: LightGBM+GRU, XGBoost+LSTM, LightGBM+LSTM, XGBoost+GRU |
| 5d | Evaluation tables, decile risk-ranking, SHAP & Integrated Gradients |

**Design choices baked in for a 16 GB Mac:**
- Snapshot features keep all 6 aggregations (mean/std/min/max/last/first) → ~1,000 ML features.
- The **sequential tensor uses only the top-15 raw features** (ranked by LightGBM gain), so the DL tensor is ~`(458913, 13, 15)` ≈ 0.35 GB instead of ~4 GB. This is what fixes the earlier *killed* crash.
- Sequential standardisation is done **column-by-column** to keep peak memory flat.
- Every modelling step prints a **timestamp + elapsed time** so you can watch progress.

Each stage **reads its inputs from disk**, so you can restart the kernel and re-run any stage on its own.


---
## Stage 0 — Environment setup

Run this once. It creates the folder structure and checks every dependency.

**Mac note (LightGBM):** LightGBM needs the OpenMP runtime. If you hit a `libomp.dylib` error, run this in a terminal once: `brew install libomp`.

If a package is missing, uncomment the `pip install` line below and run it (ideally inside a virtual environment: `python3 -m venv .venv && source .venv/bin/activate`).


In [ ]:
# Optional one-time install (uncomment if needed). Prefer running inside a venv.
# %pip install -q pandas numpy pyarrow scipy scikit-learn lightgbm xgboost torch shap captum tqdm

import sys, platform
print("Python :", sys.version.split()[0], "on", platform.platform())

missing = []
for pkg in ["pandas", "numpy", "pyarrow", "scipy", "sklearn",
            "lightgbm", "xgboost", "torch", "shap", "captum"]:
    try:
        __import__(pkg)
    except Exception as e:
        missing.append(pkg)
print("Missing packages:", missing if missing else "none ✔")
if missing:
    print("\n>>> Install them, e.g.:  %pip install -q " + " ".join(
        {'sklearn':'scikit-learn'}.get(m, m) for m in missing))


In [ ]:
# ---------------------------------------------------------------------------
# Global configuration & paths (edit here only)
# ---------------------------------------------------------------------------
from pathlib import Path
import os

# The notebook expects to live at the repository root. Adjust if you moved it.
ROOT = Path.cwd()
RAW_DIR     = ROOT / "data" / "raw"        # put train_data.csv + train_labels.csv here
PARQUET_DIR = ROOT / "data" / "parquet"
INTERIM_DIR = ROOT / "data" / "interim"
FEATURE_DIR = ROOT / "data" / "features"
MODEL_DIR   = ROOT / "data" / "models"
REPORT_DIR  = ROOT / "data" / "reports"
for d in (RAW_DIR, PARQUET_DIR, INTERIM_DIR, FEATURE_DIR, MODEL_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

TRAIN_DATA_CSV   = RAW_DIR / "train_data.csv"
TRAIN_LABELS_CSV = RAW_DIR / "train_labels.csv"

# Column groups
ID_COL, DATE_COL, TARGET_COL = "customer_ID", "S_2", "target"
CAT_COLS = ["D_63", "D_64"]                       # categoricals per the proposal slide
DROP_FEATURES = ["D_87"]                          # named drop from the slide
FEATURE_PREFIXES = {"P":"Payment","B":"Balance","S":"Spend","R":"Risk","D":"Delinquency"}

# Pipeline constants
CHUNK_SIZE        = 100_000     # rows per parquet chunk (safe for 16 GB)
MISSING_THRESHOLD = 0.90        # drop > 90% missing
DENOISE_SCALE     = 100         # multiply numeric by 100, then round
MAX_SEQ_LEN       = 13          # statements per customer
CORR_THRESHOLD    = 0.95        # drop one of any pair above this
SEQ_TOP_N         = 15          # <<< sequential tensor uses only top-15 features
NUM_AGGS          = ["mean","std","min","max","last","first"]   # 6 aggregations (kept)
CAT_AGGS          = ["last","nunique","count"]
VALID_SIZE        = 0.20
OVERFIT_GINI_DIFF = 0.30
RANDOM_STATE      = 42

# Model hyperparameters
LGB_PARAMS = dict(objective="binary", metric="binary_logloss", boosting_type="gbdt",
                  learning_rate=0.03, num_leaves=128, min_child_samples=40,
                  feature_fraction=0.30, bagging_fraction=0.70, bagging_freq=1,
                  lambda_l2=2.0, n_estimators=3000, random_state=RANDOM_STATE,
                  n_jobs=-1, verbose=-1)
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="logloss",
                  learning_rate=0.03, max_depth=6, subsample=0.80,
                  colsample_bytree=0.40, min_child_weight=8, reg_lambda=2.0,
                  n_estimators=3000, random_state=RANDOM_STATE, n_jobs=-1,
                  tree_method="hist")
DL_PARAMS = dict(hidden_size=128, num_layers=2, dropout=0.30, batch_size=1024,
                 epochs=13, lr=1e-3, weight_decay=1e-5, patience=4,
                 n_heads=8, ff_dim=256)

print("ROOT =", ROOT)
print("Looking for raw CSVs in:", RAW_DIR)
print("train_data.csv present  :", TRAIN_DATA_CSV.exists())
print("train_labels.csv present:", TRAIN_LABELS_CSV.exists())


In [ ]:
# ---------------------------------------------------------------------------
# Shared helpers: timestamped logging, memory reduction, ID compression, metrics
# ---------------------------------------------------------------------------
import time, gc, json, pickle
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, log_loss

_T0 = time.time()
def log(msg=""):
    # Print a wall-clock timestamp + elapsed-since-import prefix.
    el = time.time() - _T0
    print(f"[{datetime.now():%H:%M:%S} | +{el:6.1f}s] {msg}", flush=True)

def banner(title):
    print("\n" + "=" * 70 + f"\n{title}\n" + "=" * 70, flush=True)

def reduce_mem_usage(df, verbose=False):
    start = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
            continue
        if str(df[col].dtype).startswith("category"):
            continue
        if str(df[col].dtype).startswith("int"):
            cmin, cmax = df[col].min(), df[col].max()
            if cmin >= np.iinfo(np.int8).min and cmax <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif cmin >= np.iinfo(np.int16).min and cmax <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif cmin >= np.iinfo(np.int32).min and cmax <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        else:
            df[col] = df[col].astype(np.float32)
    if verbose:
        end = df.memory_usage(deep=True).sum() / 1024**2
        print(f"   memory {start:.1f} -> {end:.1f} MB")
    return df

def compress_customer_id(s):
    # 64-char hex customer_ID -> int64 (last 16 hex chars), ~30x smaller.
    return s.str[-16:].apply(lambda x: int(x, 16)).astype("int64")

# ---- Metrics (ROC-AUC, KS, Gini, log-loss, deciles, overfit check) ----------
def gini_from_auc(auc): return 2.0 * auc - 1.0

def ks_statistic(y_true, y_pred):
    d = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(y_pred)}).sort_values("p", ascending=False)
    cum_bad  = (d["y"].cumsum() / d["y"].sum()).to_numpy()
    cum_good = ((1 - d["y"]).cumsum() / (1 - d["y"]).sum()).to_numpy()
    return float(np.max(np.abs(cum_bad - cum_good)))

def decile_table(y_true, y_pred, n_bins=10):
    d = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(y_pred)})
    d["decile"] = pd.qcut(d["p"].rank(method="first", ascending=False),
                          q=n_bins, labels=range(1, n_bins + 1))
    g = d.groupby("decile", observed=True)
    out = pd.DataFrame({"n": g.size(), "n_bad": g["y"].sum(),
                        "default_rate": g["y"].mean(), "avg_pred_pd": g["p"].mean()}).reset_index()
    return out

def is_risk_ranked(dtable):
    r = dtable.sort_values("decile")["default_rate"].to_numpy()
    return bool(np.all(np.diff(r) <= 1e-9))

def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.clip(np.asarray(y_pred, float), 1e-15, 1 - 1e-15)
    auc = roc_auc_score(y_true, y_pred)
    dt = decile_table(y_true, y_pred)
    return {"ROC_AUC": auc, "KS": ks_statistic(y_true, y_pred), "Gini": gini_from_auc(auc),
            "LogLoss": log_loss(y_true, y_pred), "RiskRanked": is_risk_ranked(dt), "_decile": dt}

def overfit_check(tr_gini, va_gini, thr=OVERFIT_GINI_DIFF):
    if tr_gini == 0: return 0.0, False
    rel = (tr_gini - va_gini) / abs(tr_gini)
    return rel, bool(rel > thr)

def metrics_row(name, tr, va):
    rel, of = overfit_check(tr["Gini"], va["Gini"])
    return {"Model": name,
            "Train_AUC": round(tr["ROC_AUC"], 5), "Valid_AUC": round(va["ROC_AUC"], 5),
            "Train_KS": round(tr["KS"], 5), "Valid_KS": round(va["KS"], 5),
            "Train_Gini": round(tr["Gini"], 5), "Valid_Gini": round(va["Gini"], 5),
            "Train_LogLoss": round(tr["LogLoss"], 5), "Valid_LogLoss": round(va["LogLoss"], 5),
            "DiffGini_%": round(100 * rel, 2), "Overfit": of,
            "Valid_RiskRanked": va["RiskRanked"]}

log("Helpers ready.")


---
## Stage 1 — Data landing & EDA

The raw `train_data.csv` is ~16 GB, so we **stream it in 100,000-row chunks** to Parquet (compressing `customer_ID` to int64 along the way), then run EDA on the loaded Parquet:
missing rates, feature families (P/B/S/R/D), target balance, and statements-per-customer.

If the Parquet chunks already exist, landing is skipped automatically — delete `data/parquet/` to force a re-run.


In [ ]:
import glob, pyarrow  # noqa

def land_csv_to_parquet():
    if list(PARQUET_DIR.glob("train_*.parquet")):
        log("Parquet chunks already exist — skipping landing.")
        return
    if not TRAIN_DATA_CSV.exists():
        raise FileNotFoundError(f"{TRAIN_DATA_CSV} not found. Download the AMEX dataset into {RAW_DIR}.")
    banner(f"LANDING {TRAIN_DATA_CSV.name} -> Parquet (chunk={CHUNK_SIZE:,})")
    t0 = time.time()
    for i, chunk in enumerate(pd.read_csv(TRAIN_DATA_CSV, chunksize=CHUNK_SIZE)):
        chunk[ID_COL] = compress_customer_id(chunk[ID_COL])
        if DATE_COL in chunk.columns:
            chunk[DATE_COL] = pd.to_datetime(chunk[DATE_COL])
        chunk = reduce_mem_usage(chunk)
        chunk.to_parquet(PARQUET_DIR / f"train_{i:04d}.parquet", compression="zstd", index=False)
        if i % 10 == 0:
            log(f"chunk {i:>3}: {len(chunk):,} rows written")
    log(f"landing done in {time.time()-t0:.1f}s")

def load_parquet():
    files = sorted(glob.glob(str(PARQUET_DIR / "train_*.parquet")))
    return pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

land_csv_to_parquet()


In [ ]:
# Load labels (compress IDs) and the statement-level features, then run EDA.
banner("STAGE 1 — EDA")
labels = pd.read_csv(TRAIN_LABELS_CSV)
labels[ID_COL] = compress_customer_id(labels[ID_COL])
labels.to_parquet(INTERIM_DIR / "labels.parquet", index=False)

df = load_parquet()
log(f"Loaded statement-level data: {df.shape[0]:,} rows x {df.shape[1]} cols")

feat_cols = [c for c in df.columns if c not in (ID_COL, DATE_COL)]

# Missing rate + unique counts (sampled for speed on the full dataset)
samp = df[feat_cols].sample(min(200_000, len(df)), random_state=RANDOM_STATE)
missing = samp.isna().mean().sort_values(ascending=False).rename("missing_rate").to_frame()
missing["n_unique"] = [df[c].nunique(dropna=True) for c in missing.index]
missing["flag_drop"] = (missing["missing_rate"] > MISSING_THRESHOLD) | (missing["n_unique"] <= 1)
missing.to_csv(REPORT_DIR / "eda_missing.csv")

# Feature families
fam = []
for p, name in FEATURE_PREFIXES.items():
    cols = [c for c in feat_cols if c.startswith(p + "_")]
    if cols:
        fam.append({"prefix": p, "family": name, "n_features": len(cols),
                    "avg_missing_rate": float(df[cols].isna().mean().mean())})
pd.DataFrame(fam).to_csv(REPORT_DIR / "eda_feature_families.csv", index=False)

# Target balance
bal = labels[TARGET_COL].value_counts().rename("count").to_frame()
bal["pct"] = bal["count"] / bal["count"].sum()
bal.to_csv(REPORT_DIR / "eda_target_balance.csv")

counts = df.groupby(ID_COL).size()
log(f"features >90% missing : {(missing['missing_rate']>MISSING_THRESHOLD).sum()}")
log(f"constant features     : {(missing['n_unique']<=1).sum()}")
log(f"target good(0)={int(bal.loc[0,'count']):,}  bad(1)={int(bal.loc[1,'count']):,}  bad-rate={bal.loc[1,'pct']:.2%}")
log(f"statements/customer: min={counts.min()} median={int(counts.median())} max={counts.max()}")
display(pd.DataFrame(fam))
display(bal)


---
## Stage 2 — Data preprocessing

1. **Denoise** numeric features: multiply by 100 and round (recovers the quantised values hidden by AMEX's noise).
2. **Encode** the categorical variables `D_63`, `D_64` to integer codes (compact for the sequential tensor; `-1` = missing).
3. **Filter** features with > 90% missing, single unique value, or named drops (`D_87`).
4. **Sort** rows by `(customer_ID, S_2)` so each customer's statements are in chronological order.

Output: `data/interim/clean.parquet` + `kept_features.json`.


In [ ]:
banner("STAGE 2 — PREPROCESSING")
df = load_parquet()
feat_cols = [c for c in df.columns if c not in (ID_COL, DATE_COL)]
num_cols  = [c for c in feat_cols if c not in CAT_COLS]
cat_cols  = [c for c in CAT_COLS if c in df.columns]

# 2.1 Denoise numeric
log("denoising numeric features (x100, round)...")
for c in num_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        df[c] = (df[c] * DENOISE_SCALE).round(0)

# 2.1 Encode categoricals -> integer codes
log(f"encoding categoricals {cat_cols} to int codes...")
cat_categories = {}
for c in cat_cols:
    df[c] = df[c].astype("category")
    cat_categories[c] = [str(x) for x in df[c].cat.categories.tolist()]
    df[c] = df[c].cat.codes.astype("int16")

# 2.2 Filter features
log("filtering features (>90% missing | nunique<=1 | named drops)...")
drop = set(DROP_FEATURES)
miss = df[feat_cols].isna().mean();            drop |= set(miss[miss > MISSING_THRESHOLD].index)
nun  = df[feat_cols].nunique(dropna=True);     drop |= set(nun[nun <= 1].index)
drop = [c for c in drop if c in df.columns]
df = df.drop(columns=drop)
log(f"dropped {len(drop)} features: {sorted(drop)}")

kept = [c for c in df.columns if c not in (ID_COL, DATE_COL)]
kept_cat = [c for c in cat_cols if c in df.columns]
kept_num = [c for c in kept if c not in kept_cat]

# 2.3 Sort chronologically, downcast, save
df = df.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)
df = reduce_mem_usage(df)
df.to_parquet(INTERIM_DIR / "clean.parquet", compression="zstd", index=False)
with open(INTERIM_DIR / "kept_features.json", "w") as f:
    json.dump({"all": kept, "numeric": kept_num, "categorical": kept_cat,
               "cat_categories": cat_categories, "dropped": sorted(drop)}, f, indent=2)

log(f"clean.parquet: {df.shape[0]:,} rows x {df.shape[1]} cols  (numeric {len(kept_num)}, cat {len(kept_cat)})")
del df; gc.collect()


---
## Stage 3 — Feature engineering (memory-safe)

**3A · Snapshot features (ML).** Aggregate each customer's history with all **6 aggregations** (mean, std, min, max, last, first) for numeric features, plus (last, nunique, count) for categoricals, plus behavioural deltas (last−mean, last−first). → ~1,000 features, one row per customer.

**3B · Sequential features (DL).** To keep memory flat on a 16 GB Mac we:
1. Rank raw features by **LightGBM gain** (using the snapshot `_mean` columns) and keep the **top 15**.
2. Standardise those 15 columns **one at a time** (train statistics only).
3. Build the right-aligned, padded tensor `(N, 13, 15)` + a mask.

This is the change that prevents the earlier *killed* (out-of-memory) crash.


In [ ]:
banner("STAGE 3A — SNAPSHOT FEATURES (ML)")
df = pd.read_parquet(INTERIM_DIR / "clean.parquet")
labels = pd.read_parquet(INTERIM_DIR / "labels.parquet")
with open(INTERIM_DIR / "kept_features.json") as f:
    feats = json.load(f)
label_map = dict(zip(labels[ID_COL], labels[TARGET_COL]))
num, cat = feats["numeric"], feats["categorical"]

# Train/valid split BY CUSTOMER (slide: ~367k / ~92k)
rng = np.random.RandomState(RANDOM_STATE)
all_ids = labels[ID_COL].to_numpy()
perm = rng.permutation(len(all_ids)); n_valid = int(len(all_ids) * VALID_SIZE)
valid_ids = all_ids[perm[:n_valid]]; train_ids = all_ids[perm[n_valid:]]
np.savez(FEATURE_DIR / "split.npz", train_ids=train_ids, valid_ids=valid_ids)
log(f"split -> train {len(train_ids):,} | valid {len(valid_ids):,}")

log(f"aggregating {len(num)} numeric features with {NUM_AGGS}...")
num_agg = df.groupby(ID_COL)[num].agg(NUM_AGGS)
num_agg.columns = [f"{c}_{a}" for c, a in num_agg.columns]

log(f"aggregating {len(cat)} categorical features with {CAT_AGGS}...")
if cat:
    cat_agg = df.groupby(ID_COL)[cat].agg(CAT_AGGS)
    cat_agg.columns = [f"{c}_{a}" for c, a in cat_agg.columns]
else:
    cat_agg = pd.DataFrame(index=num_agg.index)

log("engineering behavioural deltas (last-mean, last-first)...")
delta = {}
for c in num:
    delta[f"{c}_last_minus_mean"]  = num_agg[f"{c}_last"]  - num_agg[f"{c}_mean"]
    delta[f"{c}_last_minus_first"] = num_agg[f"{c}_last"]  - num_agg[f"{c}_first"]
delta = pd.DataFrame(delta, index=num_agg.index)

snap = pd.concat([num_agg, cat_agg, delta], axis=1).reset_index().copy()
snap = reduce_mem_usage(snap)
snap[TARGET_COL] = snap[ID_COL].map(label_map).astype("int8")
snap.to_parquet(FEATURE_DIR / "snapshot.parquet", index=False)
log(f"snapshot: {snap.shape[0]:,} customers x {snap.shape[1]-2} features -> saved")
del num_agg, cat_agg, delta; gc.collect()


In [ ]:
banner("STAGE 3B — SEQUENTIAL FEATURES (DL), top-15 only")
import lightgbm as lgb

# 1) Rank raw features by LightGBM gain using the snapshot _mean columns.
mean_cols = [f"{c}_mean" for c in num if f"{c}_mean" in snap.columns]
tr_mask_snap = snap[ID_COL].isin(set(train_ids.tolist()))
ranker = lgb.LGBMClassifier(n_estimators=300, num_leaves=64, learning_rate=0.05,
                            feature_fraction=0.5, random_state=RANDOM_STATE,
                            n_jobs=-1, verbose=-1)
ranker.fit(snap.loc[tr_mask_snap, mean_cols], snap.loc[tr_mask_snap, TARGET_COL])
imp = pd.Series(ranker.feature_importances_, index=mean_cols).sort_values(ascending=False)
seq_feats = [c[:-5] for c in imp.head(SEQ_TOP_N).index]   # strip "_mean"
log(f"top-{SEQ_TOP_N} sequential features: {seq_feats}")
with open(FEATURE_DIR / "seq_feature_list.json", "w") as f:
    json.dump(seq_feats, f, indent=2)

# 2) Standardise the 15 columns ONE AT A TIME (train stats only) -> float32.
F, L = len(seq_feats), MAX_SEQ_LEN
tr_rows = df[ID_COL].isin(set(train_ids.tolist()))
means, stds = {}, {}
values = np.empty((len(df), F), dtype=np.float32)
for j, c in enumerate(seq_feats):
    mu = df.loc[tr_rows, c].mean()
    sd = df.loc[tr_rows, c].std() or 1.0
    means[c], stds[c] = float(mu), float(sd)
    values[:, j] = ((df[c] - mu) / sd).fillna(0).to_numpy(dtype=np.float32)
np.savez_compressed(FEATURE_DIR / "seq_scaler.npz",
                    mean=np.array(list(means.values()), np.float32),
                    std=np.array(list(stds.values()), np.float32),
                    features=np.array(seq_feats))

# 3) Build right-aligned padded tensor (N, L, F) + mask.
ids = df[ID_COL].to_numpy()
unique_ids, start_idx, counts = np.unique(ids, return_index=True, return_counts=True)
N = len(unique_ids)
X = np.zeros((N, L, F), dtype=np.float32)
Mk = np.zeros((N, L), dtype=np.float32)
log(f"building tensor ({N:,}, {L}, {F})  ~{N*L*F*4/1024**3:.2f} GB ...")
for i in range(N):
    s = start_idx[i]; n = min(counts[i], L)
    X[i, L-n:, :] = values[s + counts[i] - n : s + counts[i]]
    Mk[i, L-n:] = 1.0
y = np.array([label_map[i] for i in unique_ids], dtype=np.int8)
np.savez_compressed(FEATURE_DIR / "sequential.npz", X=X, mask=Mk, ids=unique_ids, y=y)
log(f"sequential tensor saved: X{X.shape}  mask{Mk.shape}")
del df, values, X, Mk; gc.collect()


---
## Stage 4 — Feature selection (snapshot / ML only)

Three-step funnel on the ~1,000 snapshot features:
1. **Variance** — drop near-constant columns.
2. **Correlation** — within pairs of |r| > 0.95 drop the one with more missing values.
3. **LightGBM gain** — keep features up to cumulative 99% of total gain.

The DL sequential features are kept whole (the LSTM/GRU/Transformer learn their own representation).


In [ ]:
banner("STAGE 4 — FEATURE SELECTION")
import lightgbm as lgb
snap = pd.read_parquet(FEATURE_DIR / "snapshot.parquet")
split = np.load(FEATURE_DIR / "split.npz"); train_ids = set(split["train_ids"].tolist())
cols = [c for c in snap.columns if c not in (ID_COL, TARGET_COL)]
X, y = snap[cols], snap[TARGET_COL]
log(f"start: {len(cols)} candidate features")

# 1) variance
keep = X.nunique(); keep = keep[keep > 1].index.tolist()
log(f"after variance filter   : {len(keep)}")

# 2) correlation (on a sample for speed)
samp = X[keep].sample(min(50_000, len(X)), random_state=RANDOM_STATE)
corr = samp.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
missrate = X[keep].isna().mean()
drop = set()
for col in upper.columns:
    for p in upper.index[upper[col] > CORR_THRESHOLD]:
        drop.add(p if missrate[p] >= missrate[col] else col)
keep = [c for c in keep if c not in drop]
log(f"after correlation filter: {len(keep)}  (dropped {len(drop)})")

# 3) LightGBM gain ranking
tr = snap[ID_COL].isin(train_ids)
booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=128,
                             feature_fraction=0.5, random_state=RANDOM_STATE,
                             n_jobs=-1, verbose=-1)
booster.fit(X.loc[tr, keep], y.loc[tr])
imp = pd.Series(booster.feature_importances_, index=keep, name="gain").sort_values(ascending=False)
imp.to_csv(REPORT_DIR / "feature_importance_ranking.csv")
cum = imp.cumsum() / imp.sum()
selected = cum[cum <= 0.99].index.tolist() or imp.index[:50].tolist()
log(f"after gain ranking      : {len(selected)}  (cumulative 99% gain)")

snap[[ID_COL] + selected + [TARGET_COL]].to_parquet(FEATURE_DIR / "snapshot_selected.parquet", index=False)
with open(FEATURE_DIR / "selected_features.json", "w") as f:
    json.dump(selected, f, indent=2)
log("selected snapshot saved.")
display(imp.head(15).to_frame("gain"))


---
## Stage 5a — Machine-learning models

Logistic Regression, XGBoost and LightGBM on the selected snapshot features. Each model is timestamped, scored on train + validate with the shared metrics, and its per-customer PD is saved for the ensembles.


In [ ]:
banner("STAGE 5A — ML MODELS")
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb, xgboost as xgb

snap = pd.read_parquet(FEATURE_DIR / "snapshot_selected.parquet")
split = np.load(FEATURE_DIR / "split.npz"); train_ids = set(split["train_ids"].tolist())
feats = [c for c in snap.columns if c not in (ID_COL, TARGET_COL)]
tr = snap[ID_COL].isin(train_ids)
Xtr, ytr = snap.loc[tr, feats], snap.loc[tr, TARGET_COL]
Xva, yva = snap.loc[~tr, feats], snap.loc[~tr, TARGET_COL]
log(f"train {len(Xtr):,} | valid {len(Xva):,} | features {len(feats)}")

rows = []; preds = {ID_COL: snap[ID_COL], "split": np.where(tr, "train", "valid")}
def record(name, p_tr, p_va, model):
    ev_tr, ev_va = evaluate(ytr, p_tr), evaluate(yva, p_va)
    rows.append(metrics_row(name, ev_tr, ev_va))
    full = np.empty(len(snap)); full[tr.to_numpy()] = p_tr; full[~tr.to_numpy()] = p_va
    preds[name] = full
    with open(MODEL_DIR / f"{name}.pkl", "wb") as f: pickle.dump(model, f)
    log(f"{name:<20} valid AUC={ev_va['ROC_AUC']:.4f}  Gini={ev_va['Gini']:.4f}  KS={ev_va['KS']:.4f}")

log("training Logistic Regression...")
logreg = Pipeline([("impute", SimpleImputer(strategy="median")),
                   ("scale", StandardScaler()),
                   ("clf", LogisticRegression(max_iter=2000))]).fit(Xtr, ytr)
record("LogisticRegression", logreg.predict_proba(Xtr)[:,1], logreg.predict_proba(Xva)[:,1], logreg)

log("training XGBoost...")
xgbm = xgb.XGBClassifier(**XGB_PARAMS)
xgbm.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
record("XGBoost", xgbm.predict_proba(Xtr)[:,1], xgbm.predict_proba(Xva)[:,1], xgbm)

log("training LightGBM...")
lgbm = lgb.LGBMClassifier(**LGB_PARAMS)
lgbm.fit(Xtr, ytr, eval_set=[(Xva, yva)],
         callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
record("LightGBM", lgbm.predict_proba(Xtr)[:,1], lgbm.predict_proba(Xva)[:,1], lgbm)

pd.DataFrame(preds).to_parquet(FEATURE_DIR / "preds_ml.parquet", index=False)
pd.DataFrame(rows).to_csv(REPORT_DIR / "metrics_ml.csv", index=False)
log("Stage 5a complete."); display(pd.DataFrame(rows))


---
## Stage 5b — Deep-learning models (LSTM, GRU, Transformer)

Trains on the `(N, 13, 15)` sequential tensor for **13 epochs** each. Device auto-detects **MPS (Apple M4 GPU)** → CUDA → CPU. Every epoch prints a **timestamp, elapsed time, and validation AUC**, with early stopping on validation AUC.


In [ ]:
banner("STAGE 5B — DEEP-LEARNING MODELS")
import math, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
device = get_device(); log(f"device: {device}")

class RecurrentClassifier(nn.Module):
    def __init__(self, n_feat, cell="LSTM", hidden=128, layers=2, dropout=0.3):
        super().__init__()
        rnn = nn.LSTM if cell == "LSTM" else nn.GRU
        self.rnn = rnn(n_feat, hidden, num_layers=layers, batch_first=True,
                       dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden//2),
                                  nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden//2, 1))
    def forward(self, x, mask):
        out, _ = self.rnn(x)
        lengths = mask.sum(1).long().clamp(min=1)
        last = out[torch.arange(out.size(0)), lengths - 1]
        return self.head(last).squeeze(-1)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[: pe[:, 1::2].size(1)])
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, : x.size(1)]

class TransformerClassifier(nn.Module):
    def __init__(self, n_feat, hidden=128, layers=2, heads=8, ff=256, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(n_feat, hidden)
        self.pos = PositionalEncoding(hidden, MAX_SEQ_LEN + 1)
        enc = nn.TransformerEncoderLayer(hidden, heads, ff, dropout,
                                         batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc, layers)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden//2),
                                  nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden//2, 1))
    def forward(self, x, mask):
        h = self.pos(self.proj(x))
        h = self.encoder(h, src_key_padding_mask=(mask == 0))
        m = mask.unsqueeze(-1)
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return self.head(pooled).squeeze(-1)

# --- load tensor + split indices
data = np.load(FEATURE_DIR / "sequential.npz")
X, Mk, ids, y = data["X"], data["mask"], data["ids"], data["y"].astype(int)
split = np.load(FEATURE_DIR / "split.npz"); train_set = set(split["train_ids"].tolist())
tr_idx = np.where(np.isin(ids, list(train_set)))[0]
va_idx = np.where(~np.isin(ids, list(train_set)))[0]
F = X.shape[2]; log(f"X{X.shape}  train {len(tr_idx):,} | valid {len(va_idx):,}")

def predict(model, Xa, Ma, batch=4096):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(Xa), batch):
            xb = torch.tensor(Xa[i:i+batch]).to(device); mb = torch.tensor(Ma[i:i+batch]).to(device)
            out.append(torch.sigmoid(model(xb, mb)).cpu().numpy())
    return np.concatenate(out)

def train_model(name, model):
    p = DL_PARAMS; model = model.to(device)
    pos_w = torch.tensor([(y[tr_idx]==0).sum() / max((y[tr_idx]==1).sum(),1)],
                         dtype=torch.float32, device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=p["lr"], weight_decay=p["weight_decay"])
    ds = TensorDataset(torch.tensor(X[tr_idx]), torch.tensor(Mk[tr_idx]),
                       torch.tensor(y[tr_idx], dtype=torch.float32))
    loader = DataLoader(ds, batch_size=p["batch_size"], shuffle=True)
    best_auc, best_state, wait = -1, None, 0
    log(f"=== training {name} for {p['epochs']} epochs ===")
    for epoch in range(p["epochs"]):
        model.train(); t0 = time.time()
        for xb, mb, yb in loader:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(xb, mb), yb).backward(); opt.step()
        auc = roc_auc_score(y[va_idx], predict(model, X[va_idx], Mk[va_idx]))
        log(f"{name} epoch {epoch+1:>2}/{p['epochs']}  valid AUC={auc:.4f}  ({time.time()-t0:.1f}s)")
        if auc > best_auc:
            best_auc, wait = auc, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= p["patience"]:
                log(f"{name} early stop at epoch {epoch+1}"); break
    if best_state: model.load_state_dict(best_state)
    return model

p = DL_PARAMS
builders = {
    "LSTM":        lambda: RecurrentClassifier(F, "LSTM", p["hidden_size"], p["num_layers"], p["dropout"]),
    "GRU":         lambda: RecurrentClassifier(F, "GRU",  p["hidden_size"], p["num_layers"], p["dropout"]),
    "Transformer": lambda: TransformerClassifier(F, p["hidden_size"], p["num_layers"], p["n_heads"], p["ff_dim"], p["dropout"]),
}
rows = []; preds = {ID_COL: ids, "split": np.where(np.isin(ids, list(train_set)), "train", "valid")}
for name, build in builders.items():
    torch.manual_seed(RANDOM_STATE)
    model = train_model(name, build())
    torch.save(model.state_dict(), MODEL_DIR / f"{name.lower()}.pt")
    p_tr, p_va = predict(model, X[tr_idx], Mk[tr_idx]), predict(model, X[va_idx], Mk[va_idx])
    full = np.empty(len(ids)); full[tr_idx] = p_tr; full[va_idx] = p_va; preds[name] = full
    rows.append(metrics_row(name, evaluate(y[tr_idx], p_tr), evaluate(y[va_idx], p_va)))
    log(f"{name} done -> valid Gini={rows[-1]['Valid_Gini']:.4f}")

pd.DataFrame(preds).to_parquet(FEATURE_DIR / "preds_dl.parquet", index=False)
pd.DataFrame(rows).to_csv(REPORT_DIR / "metrics_dl.csv", index=False)
log("Stage 5b complete."); display(pd.DataFrame(rows))


---
## Stage 5c — Ensembles (ML + DL)

Four hybrids — **LightGBM+GRU, XGBoost+LSTM, LightGBM+LSTM, XGBoost+GRU** — built as a rank-average blend `p = w·p_tree + (1−w)·p_seq`, where `w` is chosen on the validation set to maximise Gini, then applied to both splits.


In [ ]:
banner("STAGE 5C — ENSEMBLES")
from scipy.stats import rankdata
ENSEMBLES = [("LightGBM+GRU","LightGBM","GRU"), ("XGBoost+LSTM","XGBoost","LSTM"),
             ("LightGBM+LSTM","LightGBM","LSTM"), ("XGBoost+GRU","XGBoost","GRU")]

ml = pd.read_parquet(FEATURE_DIR / "preds_ml.parquet")
dl = pd.read_parquet(FEATURE_DIR / "preds_dl.parquet")
y = pd.read_parquet(INTERIM_DIR / "labels.parquet")
df = ml.merge(dl.drop(columns=["split"]), on=ID_COL).merge(y, on=ID_COL)
tr = df["split"].to_numpy() == "train"
ytr, yva = df.loc[tr, TARGET_COL].to_numpy(), df.loc[~tr, TARGET_COL].to_numpy()

def to_rank(p): return rankdata(p) / len(p)
def best_w(yv, pt, ps):
    rt, rs = to_rank(pt), to_rank(ps); bw, bg = 0.5, -1
    for w in np.linspace(0, 1, 21):
        g = evaluate(yv, w*rt + (1-w)*rs)["Gini"]
        if g > bg: bg, bw = g, w
    return bw

rows = []; preds = {ID_COL: df[ID_COL], "split": df["split"]}
for name, tree, seq in ENSEMBLES:
    w = best_w(yva, df.loc[~tr, tree].to_numpy(), df.loc[~tr, seq].to_numpy())
    blend = lambda m: w*to_rank(df.loc[m, tree].to_numpy()) + (1-w)*to_rank(df.loc[m, seq].to_numpy())
    p_tr, p_va = blend(tr), blend(~tr)
    full = np.empty(len(df)); full[tr] = p_tr; full[~tr] = p_va; preds[name] = full
    rows.append(metrics_row(name, evaluate(ytr, p_tr), evaluate(yva, p_va)))
    log(f"{name:<16} w(tree)={w:.2f}  valid Gini={rows[-1]['Valid_Gini']:.4f}")

pd.DataFrame(preds).to_parquet(FEATURE_DIR / "preds_ensemble.parquet", index=False)
pd.DataFrame(rows).to_csv(REPORT_DIR / "metrics_ensemble.csv", index=False)
log("Stage 5c complete."); display(pd.DataFrame(rows))


---
## Stage 5d — Evaluation, comparison & interpretability

Builds the two slide-style tables (train + validate with **Diff Gini %** overfit flag), the decile risk-ranking of the best model, then the interpretability artefacts: **SHAP** (best tree model) and **Integrated Gradients** (LSTM).


In [ ]:
banner("STAGE 5D — EVALUATION & COMPARISON")
parts = [pd.read_csv(REPORT_DIR / f) for f in
         ["metrics_ml.csv","metrics_dl.csv","metrics_ensemble.csv"] if (REPORT_DIR / f).exists()]
allm = pd.concat(parts, ignore_index=True)

train_tbl = allm[["Model","Train_AUC","Train_KS","Train_Gini","Train_LogLoss"]].copy()
train_tbl.columns = ["Model","ROC_AUC","KS","Gini","LogLoss"]
valid_tbl = allm[["Model","Valid_AUC","Valid_KS","Valid_Gini","Valid_LogLoss",
                  "DiffGini_%","Overfit","Valid_RiskRanked"]].copy()
valid_tbl.columns = ["Model","ROC_AUC","KS","Gini","LogLoss","DiffGini_%","Overfit","RiskRanked"]
train_tbl = train_tbl.sort_values("Gini", ascending=False).reset_index(drop=True)
valid_tbl = valid_tbl.sort_values("Gini", ascending=False).reset_index(drop=True)
train_tbl.to_csv(REPORT_DIR / "comparison_train.csv", index=False)
valid_tbl.to_csv(REPORT_DIR / "comparison_validate.csv", index=False)

print("\nTABLE 1 — TRAINING SET");  display(train_tbl)
print("\nTABLE 2 — VALIDATE SET (overfit check)"); display(valid_tbl)

# Decile risk-ranking of the best validation model
best = valid_tbl.iloc[0]["Model"]
for f in ["preds_ml.parquet","preds_dl.parquet","preds_ensemble.parquet"]:
    d = pd.read_parquet(FEATURE_DIR / f)
    if best in d.columns: break
yy = pd.read_parquet(INTERIM_DIR / "labels.parquet")
d = d.merge(yy, on=ID_COL); va = d[d["split"] == "valid"]
dt = decile_table(va[TARGET_COL].to_numpy(), va[best].to_numpy())
dt.to_csv(REPORT_DIR / "best_model_deciles.csv", index=False)
log(f"best validation model: {best}  | risk-ranked: {evaluate(va[TARGET_COL], va[best])['RiskRanked']}")
display(dt)


In [ ]:
# Interpretability — SHAP (best tree model) + Integrated Gradients (LSTM)
banner("STAGE 5D — INTERPRETABILITY")

# SHAP on LightGBM
try:
    import shap
    snap = pd.read_parquet(FEATURE_DIR / "snapshot_selected.parquet")
    feats = [c for c in snap.columns if c not in (ID_COL, TARGET_COL)]
    with open(MODEL_DIR / "LightGBM.pkl", "rb") as f: model = pickle.load(f)
    sample = snap[feats].sample(min(5000, len(snap)), random_state=RANDOM_STATE)
    sv = shap.TreeExplainer(model).shap_values(sample)
    sv = sv[1] if isinstance(sv, list) else sv
    shap_imp = pd.Series(np.abs(sv).mean(0), index=feats).sort_values(ascending=False).head(10)
    shap_imp.to_csv(REPORT_DIR / "shap_top10.csv")
    log("SHAP top-10 (LightGBM):"); display(shap_imp.to_frame("mean_abs_shap"))
except Exception as e:
    log(f"SHAP skipped: {e}")

# Integrated Gradients on LSTM
try:
    import torch
    from captum.attr import IntegratedGradients
    data = np.load(FEATURE_DIR / "sequential.npz"); X, Mk = data["X"], data["mask"]
    seq_feats = json.load(open(FEATURE_DIR / "seq_feature_list.json"))
    lstm = RecurrentClassifier(X.shape[2], "LSTM", DL_PARAMS["hidden_size"],
                               DL_PARAMS["num_layers"], DL_PARAMS["dropout"]).to(device)
    lstm.load_state_dict(torch.load(MODEL_DIR / "lstm.pt", map_location=device)); lstm.eval()
    idx = np.random.RandomState(RANDOM_STATE).choice(len(X), size=min(256, len(X)), replace=False)
    xb = torch.tensor(X[idx]).to(device); mb = torch.tensor(Mk[idx]).to(device)
    attr = IntegratedGradients(lstm).attribute(xb, baselines=torch.zeros_like(xb),
                                                additional_forward_args=(mb,), n_steps=16,
                                                internal_batch_size=64)
    ig_imp = pd.Series(attr.abs().mean(dim=(0,1)).detach().cpu().numpy(),
                       index=seq_feats).sort_values(ascending=False).head(10)
    ig_imp.to_csv(REPORT_DIR / "ig_top10.csv")
    log("Integrated Gradients top-10 (LSTM):"); display(ig_imp.to_frame("mean_abs_ig"))
except Exception as e:
    log(f"Integrated Gradients skipped: {e}")


---
## Final summary & pushing to GitLab

All result tables are in `data/reports/`:
`comparison_train.csv`, `comparison_validate.csv`, `best_model_deciles.csv`, `shap_top10.csv`, `ig_top10.csv`, plus the EDA tables. Use these directly in your slides / thesis.

### Export this notebook for your presentation
- VS Code: open the Command Palette → **Jupyter: Export to HTML / PDF**, or
- Terminal: `jupyter nbconvert --to html amex_pd_pipeline.ipynb`

### Push the code to GitLab
The raw data and generated artefacts are git-ignored (see `.gitignore`) — only code, the notebook and the small report CSVs are committed.

```bash
git init
git add .
git commit -m "AMEX PD thesis: full local pipeline (ML + DL + ensembles)"
git branch -M main
git remote add origin https://gitlab.com/<your-username>/<your-repo>.git
git push -u origin main
```

If GitLab asks for credentials, use a **Personal Access Token** (GitLab → Settings → Access Tokens, scope `write_repository`) as the password, or set up an SSH key. If you ever add a wrong remote: `git remote remove origin` then re-add the correct URL.


In [ ]:
# One-cell recap of everything produced (handy screenshot for your slides)
banner("PIPELINE SUMMARY")
for name, path in [
    ("Parquet chunks (Stage 1)", PARQUET_DIR),
    ("Clean data (Stage 2)",     INTERIM_DIR / "clean.parquet"),
    ("Snapshot ML (Stage 3A)",   FEATURE_DIR / "snapshot.parquet"),
    ("Sequential DL (Stage 3B)", FEATURE_DIR / "sequential.npz"),
    ("Selected snapshot (St 4)", FEATURE_DIR / "snapshot_selected.parquet"),
    ("Comparison (validate)",    REPORT_DIR / "comparison_validate.csv"),
]:
    if Path(path).exists():
        if Path(path).is_dir():
            print(f"  ✔ {name:<26} {len(list(Path(path).iterdir()))} files")
        else:
            print(f"  ✔ {name:<26} {Path(path).stat().st_size/1024**2:.1f} MB")
    else:
        print(f"  x  {name:<26} (run its stage)")
log("Done.")
